<a href="https://colab.research.google.com/github/Sachishah21/ai-patient-intelligence-system/blob/main/hospital_ai_copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Project Overview

In [ ]:
"""
Hospital AI Copilot

This system:
1. Ingests patient PDFs containing ground-truth history and multi-visit handovers
2. Extracts patient safety signals directly from clinical text
3. Visualizes:
   - Patient stability trend
   - Medical history documentation consistency
   - Medication–allergy conflict events
4. Augments decision-making using historical patient cases (RAG)
5. Provides role-specific AI copilots:
   - Doctor 1: Treating doctor
   - Doctor 2: Incoming / handover doctor

Decision-support only. No diagnosis.
"""


###CODE

Installing Dependencies

In [ ]:
!pip install PyPDF2 pandas matplotlib seaborn groq sentence-transformers faiss-cpu

Importing libraries

In [ ]:
import os
import re
import json
import PyPDF2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from groq import Groq
from google.colab import files, userdata
from sentence_transformers import SentenceTransformer
import faiss

API KEY Calling

In [ ]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("Groq API loaded")

Uploading 3 files

In [ ]:
!git clone https://github.com/Sachishah21/ai-patient-intelligence-system.git

import os
os.chdir("ai-patient-intelligence-system")

files = [
    "data/Patient_1.pdf",
    "data/Patient_2.pdf",
    "data/Patient 3.pdf"
]

for f in files:
    print(f, os.path.exists(f))

In [ ]:
def extract_text(pdf_path):
    reader = PyPDF2.PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

pdf_texts = {fname: extract_text(fname) for fname in files}


VISUALIZATIONS

In [ ]:
for fname, text in pdf_texts.items():

    # ---------- Split visits ----------
    visit_blocks = re.split(r"Visit\s*\d+", text, flags=re.IGNORECASE)[1:]
    visit_labels = [f"Visit {i+1}" for i in range(len(visit_blocks))]

    # ---------- 1️⃣ Stability Trend ----------
    stability_scores = [
        int(s) for s in re.findall(
            r"stability\s*score\s*:\s*(\d+)",
            text,
            flags=re.IGNORECASE
        )
    ]
    if len(stability_scores) < len(visit_labels):
        stability_scores += [None] * (len(visit_labels) - len(stability_scores))

    # ---------- 2️⃣ Medical History Consistency ----------
    medical_keys = {
        "Diabetes": "diabetes",
        "Hypertension": "hypertension",
        "Chronic Kidney Disease": "kidney",
        "Drug Allergy": "allergy"
    }

    consistency_data = {
        visit_labels[i]: [
            int(key in visit.lower()) for key in medical_keys.values()
        ]
        for i, visit in enumerate(visit_blocks)
    }

    consistency_df = pd.DataFrame(
        consistency_data,
        index=medical_keys.keys()
    )

    # ---------- 3️⃣ Clinical Risk Burden ----------
    risk_burden = []

    for i, visit in enumerate(visit_blocks):
        visit_lower = visit.lower()
        risk_score = 0

        # Chronic conditions
        if "diabetes" in visit_lower:
            risk_score += 1
        if "hypertension" in visit_lower:
            risk_score += 1
        if "kidney" in visit_lower:
            risk_score += 1

        # Allergy risk
        if "allergy" in visit_lower:
            risk_score += 1

        # Social risk
        if "smoker" in visit_lower or "smoking" in visit_lower:
            risk_score += 1

        # Stability deterioration
        if i > 0 and stability_scores[i] is not None and stability_scores[i-1] is not None:
            if stability_scores[i] < stability_scores[i-1]:
                risk_score += 1

        risk_burden.append(risk_score)

    # ---------- Plot Subplots ----------
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    fig.suptitle(f"Patient Safety Overview — {fname}", fontsize=14)

    # Plot 1: Stability
    axes[0].plot(visit_labels, stability_scores, marker="o", linewidth=2)
    axes[0].set_ylim(0, 10)
    axes[0].set_title("Patient Stability Trend")
    axes[0].set_ylabel("Stability Score")
    axes[0].grid(True, linestyle="--", alpha=0.6)

    # Plot 2: Consistency Heatmap
    sns.heatmap(
        consistency_df,
        annot=True,
        cmap="RdYlGn",
        cbar=False,
        linewidths=0.5,
        ax=axes[1]
    )
    axes[1].set_title("Medical History Documentation Consistency")

    # Plot 3: Risk Burden
    axes[2].bar(
        visit_labels,
        risk_burden,
        color="orange",
        edgecolor="black"
    )
    axes[2].set_title("Clinical Risk Burden per Visit")
    axes[2].set_ylabel("Number of Risk Factors")
    axes[2].set_ylim(0, max(risk_burden) + 1)

    plt.tight_layout()
    plt.show()


100 Patient Database

In [ ]:
import pandas as pd
import random

random.seed(42)

primary_conditions = [
    "Type 2 Diabetes Mellitus",
    "Hypertension",
    "Chronic Kidney Disease",
    "Ischemic Heart Disease",
    "Sepsis",
    "Pneumonia"
]

comorbidities = [
    "Hypertension",
    "Chronic Kidney Disease",
    "Obesity",
    "Asthma",
    "Ischemic Heart Disease",
    "None"
]

allergies = [
    ("Sulfa drugs", "severe rash"),
    ("Penicillin", "anaphylaxis"),
    ("NSAIDs", "gastric bleed"),
    ("None", None)
]

lessons = [
    "Proper allergy documentation prevented adverse drug reactions.",
    "Incomplete handover led to delayed recognition of allergy.",
    "Early specialist consultation improved outcomes.",
    "Medication reconciliation reduced complications.",
    "Poor documentation increased patient safety risk."
]

historical_patients = []

for i in range(1, 101):
    age = random.randint(45, 80)
    gender = random.choice(["male", "female"])
    primary = random.choice(primary_conditions)
    comorb = random.choice(comorbidities)
    allergy, reaction = random.choice(allergies)
    outcome = random.randint(3, 9)
    lesson = random.choice(lessons)

    summary = f"""
{age}-year-old {gender} admitted with {primary}.
Comorbidities included {comorb}.
"""

    if allergy != "None":
        summary += f"Documented allergy to {allergy} causing {reaction}. "
    else:
        summary += "No known drug allergies at admission. "

    summary += f"""
Treatment decisions varied based on handover quality.
Outcome score {outcome}/10.
Key lesson: {lesson}
"""

    historical_patients.append({
        "Patient_ID": f"HIST_{i:03d}",
        "Patient_Summary": summary.strip()
    })

historical_df = pd.DataFrame(historical_patients)

# Save CSV
historical_df.to_csv("historical_patients.csv", index=False)

print("✅ 100 historical patient records created")
historical_df.head()


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

historical_texts = historical_df['Patient_Summary'].tolist()
embeddings = embedder.encode(historical_texts, show_progress_bar=True)

faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
faiss_index.add(np.array(embeddings))

print("✅ Historical patient RAG index built")

In [ ]:
def get_similarity_tips(current_patient_text, top_k=3):
    query_embedding = embedder.encode([current_patient_text])
    _, indices = faiss_index.search(query_embedding, top_k)

    similar_cases = [historical_texts[i] for i in indices[0]]

    tips_prompt = f"""
You are a clinical decision-support assistant.

CURRENT PATIENT SUMMARY:
{current_patient_text}

SIMILAR HISTORICAL PATIENT CASES:
{chr(10).join(similar_cases)}

Task:
- Identify what worked well in similar patients
- Identify what went wrong in similar patients
- Give 3 concise safety-focused tips for the doctor
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": "You are a medical safety assistant."},
            {"role": "user", "content": tips_prompt}
        ]
    )

    return response.choices[0].message.content

In [ ]:
def get_similarity_context(current_patient_text, top_k=3):
    query_embedding = embedder.encode([current_patient_text])
    _, indices = faiss_index.search(query_embedding, top_k)

    similar_cases = [historical_texts[i] for i in indices[0]]

    tips_prompt = f"""
You are a clinical decision-support assistant.

CURRENT PATIENT SUMMARY:
{current_patient_text}

SIMILAR HISTORICAL PATIENT CASES:
{chr(10).join(similar_cases)}

Task:
- Identify what worked well
- Identify what went wrong
- Generate concise safety-focused tips
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": "You are a medical safety assistant."},
            {"role": "user", "content": tips_prompt}
        ]
    )

    tips = response.choices[0].message.content

    return similar_cases, tips


In [ ]:
doctor1_chat_history = []
doctor2_chat_history = []

In [ ]:
def doctor1_chatbot(question):
    prompt = f"""
You are Doctor 1's AI Copilot.

ROLE:
- Treating doctor during active shift

CURRENT PATIENT DATA:
{current_patient_corpus}

SIMILAR PAST PATIENTS (FROM HISTORY DATABASE):
{chr(10).join(similar_cases)}

LESSONS FROM SIMILAR PATIENTS:
{similarity_tips}

QUESTION:
{question}

Rules:
- No diagnosis
- Clearly reference similar past patients when relevant
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": "You assist the treating doctor."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [ ]:
def doctor2_chatbot(question):
    prompt = f"""
You are Doctor 2's AI Copilot.

ROLE:
- Incoming doctor during handover

CURRENT PATIENT DATA:
{current_patient_corpus}

SIMILAR PAST PATIENTS (FROM HISTORY DATABASE):
{chr(10).join(similar_cases)}

LESSONS FROM SIMILAR PATIENTS:
{similarity_tips}

QUESTION:
{question}

Rules:
- No diagnosis
- Highlight risks seen in similar past patients
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": "You assist the incoming doctor."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [ ]:
current_patient_corpus = "\n\n".join(pdf_texts.values())

similar_cases, similarity_tips = get_similarity_context(current_patient_corpus)

print("✅ Similar historical patients retrieved:")
for i, case in enumerate(similar_cases, 1):
    print(f"\nSimilar Patient {i}:\n{case[:300]}...")


In [ ]:
current_patient_text = "\n\n".join(pdf_texts.values())
similar_cases, similarity_tips = get_similarity_context(current_patient_text)
print("🔍 Similar Historical Patients Retrieved:\n")

for i, case in enumerate(similar_cases, 1):
    print(f"--- Similar Patient {i} ---")
    print(case)
    print()
print("🧠 Safety Tips Based on Similar Patients:\n")
print(similarity_tips)

In [ ]:
print("\n🏥 Hospital AI Copilot")
print("Choose your role:")
print("1 → Doctor 1 (Treating Doctor)")
print("2 → Doctor 2 (Incoming / Handover Doctor)")
print("Type 'exit' to quit\n")

while True:
    role = input("Select role (1/2): ").strip()

    if role.lower() == "exit":
        print("Session ended.")
        break

    if role not in ["1", "2"]:
        print("Invalid choice.")
        continue

    question = input("Ask your question: ").strip()

    if question.lower() in ["exit", "quit"]:
        print("Session ended.")
        break

    if role == "1":
        print("\nDoctor 1 AI:", doctor1_chatbot(question), "\n")
    else:
        print("\nDoctor 2 AI:", doctor2_chatbot(question), "\n")